In [1]:
# activity data
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient

In [2]:
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

Activity.csv
Demographics.csv
Labels.csv
Physiology.csv
Sleep.csv


In [3]:
sleep_df1 = gen_date_col(data_dict['Sleep'],tcol='timestamp')
sleep_df1

,patient_id,timestamp,state,heart_rate,respiratory_rate,snoring,date
0,0f352,2019-06-25 22:53:00,AWAKE,69.0,14.0,False,2019-06-25
1,0f352,2019-06-25 22:54:00,AWAKE,66.0,14.0,False,2019-06-25
2,0f352,2019-06-25 22:55:00,AWAKE,70.0,14.0,False,2019-06-25
3,0f352,2019-06-25 22:56:00,AWAKE,70.0,13.0,False,2019-06-25
4,0f352,2019-06-25 22:57:00,AWAKE,68.0,13.0,False,2019-06-25
...,...,...,...,...,...,...,...
461418,f220c,2019-06-30 10:47:00,AWAKE,76.0,20.0,False,2019-06-30
461419,f220c,2019-06-30 10:48:00,AWAKE,73.0,21.0,False,2019-06-30
461420,f220c,2019-06-30 10:49:00,AWAKE,65.0,18.0,False,2019-06-30
461421,f220c,2019-06-30 10:50:00,AWAKE,75.0,15.0,False,2019-06-30


In [4]:
sleep_id_array = sleep_df1.patient_id.unique()
sleep_id_array

['0f352', '16f4b', '1fbe4', '30a32', '55cd4', ..., 'c8574', 'd7a46', 'e2472', 'ec812', 'f220c']
Length: 17
Categories (17, object): ['0f352', '16f4b', '1fbe4', '30a32', ..., 'd7a46', 'e2472', 'ec812', 'f220c']

In [6]:

df_agitation_raw = pd.read_csv('../Dataset/Labels.csv')
df_agitation = df_agitation_raw.query(' type == "Agitation" ').copy()
df_agitation['date'] = pd.to_datetime(df_agitation['date'])
df_agitation['dates'] = df_agitation['date'].dt.date
df_agitation

,patient_id,date,type,dates
2,16f4b,2019-04-11 12:00:22,Agitation,2019-04-11
4,16f4b,2019-04-14 12:00:07,Agitation,2019-04-14
5,16f4b,2019-04-15 18:00:24,Agitation,2019-04-15
6,16f4b,2019-04-16 18:00:38,Agitation,2019-04-16
11,16f4b,2019-04-21 12:00:55,Agitation,2019-04-21
...,...,...,...,...
587,0d5ef,2019-06-28 18:02:11,Agitation,2019-06-28
588,d7a46,2019-06-28 18:02:21,Agitation,2019-06-28
598,6b29b,2019-06-29 12:00:27,Agitation,2019-06-29
599,95899,2019-06-29 12:01:20,Agitation,2019-06-29


# visulization of sleep states with agitation marks

In [13]:
import pandas as pd
import datetime
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Map states to heights and colors
state_mapping = {
    'AWAKE': {'height': 0, 'color': 'darkorange'},  
    'LIGHT': {'height': 1, 'color': 'lightsteelblue'},
    'DEEP': {'height': 2, 'color': 'royalblue'},
    'REM': {'height': 3, 'color': 'navy'}
}


summary_df_list =[]

for id_pick in sleep_id_array:
    print(id_pick)
    import os

   ########### create output folder
    folder_path = f'output/sleep_states_agitation/{id_pick}'    
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
      
    sleep_df_idPick = sleep_df1[sleep_df1['patient_id'] == id_pick] 
    df_agitation_idPick = df_agitation[df_agitation['patient_id'] == id_pick]   
    
    date_array = sleep_df_idPick.date.unique()

    date_list = []
    bout_number_list = []
    bout_duration_list = []
    bout_duration_avg_list = []
    bout_duration_med_list = []
    
    # for target_date in date_array:
    #     date_list.append(target_date)
        # in each day
    # filtered_df = sleep_df_idPick[sleep_df_idPick['date'] == target_date]
    filtered_df = sleep_df_idPick.copy()
    
    # Assuming your dataframe is called sleep_df_idPick
    # First, make sure timestamp is a datetime object
    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
    
    
    # Sort by timestamp
    filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
          
    
##################### calculate sleep duration ##########################
    # Identify chunks of consecutive same state
    filtered_df['state_change'] = (filtered_df['state'] != filtered_df['state'].shift()).cumsum()
    
    # Group by each chunk
    chunk_durations = filtered_df.groupby(['state_change', 'state']).agg(
        start_time=('timestamp', 'first'),
        end_time=('timestamp', 'last')
    ).reset_index()
    chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
    
    # Calculate duration in minutes per chunk
    chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60

################ segment long dataset to 7-day chunks    
    chunk_durations['dates'] = chunk_durations['start_time'].dt.date 
    dates = chunk_durations.dates.unique()
    
    def split_into_chunks(dates, chunk_size=7):
        return [dates[i:i + chunk_size] for i in range(0, len(dates), chunk_size)]
    
    # Split the dates into chunks of 7
    chunks = split_into_chunks(dates)

    ################## plot sleep state
    chunk_n = 0
    for select_date in chunks:
        chunk_n += 1
        chunk_durations_split = chunk_durations[chunk_durations['dates'].isin(select_date)]
        # Set the day offset for multiple days (e.g., 5 units of height between days)
        day_offset = 5
        
        # Prepare a list for plotting
        plot_data = []
        # the dates in the subset data -- chunk_durations_split
        select_date_list = select_date.tolist()
        for index, row in chunk_durations_split.iterrows():
            duration_hour = row['duration_min']/ 60
            # delete abnormal data (with lots of time missing in neighbor states)
            if duration_hour < 5:
    
                day_number = select_date_list.index(row['dates']) 
    
                start_hour = row['start_time'].hour + row['start_time'].minute / 60
                end_hour = row['end_time'].hour + row['end_time'].minute / 60
                
                state_info = state_mapping[row['state']]
                
                # Adjust the y position based on the day number
                y_position = state_info['height'] + day_number * day_offset
                
                plot_data.append({
                    'start_hour': start_hour,
                    'end_hour': end_hour,
                    'duration':duration_hour,
                    'height': state_info['height'],
                    'color': state_info['color'],
                    'y_position': y_position
                })
        # Prepare a list for plotting (agitation)
        agitation_data = []
        for index, row in df_agitation_idPick.iterrows():
            if row['dates'] in select_date_list:          
                day_number = select_date_list.index(row['dates']) 
                
                start_hour = row['date'].hour + row['date'].minute / 60
                # Adjust the y position based on the day number
                y_position = (day_number * day_offset) / (len(select_date_list)*5)
                y_gap = 1/len(select_date)
                agitation_data.append({
                    'start_hour': start_hour,
                    'y_position': y_position,
                    'y_gap':y_gap
                })    
    
     
        # Plot
        
        plt.figure(figsize=(10, 6))
        
        
        for state in plot_data:
            plt.barh(y=state['y_position'], left=state['start_hour'], width=state['duration'], height=0.8, color=state['color'])
        
        for agitation_mark in agitation_data:
            plt.axvline(x=agitation_mark['start_hour'], ymin=agitation_mark['y_position'] , ymax= agitation_mark['y_position'] + agitation_mark['y_gap'] , color='red', linestyle='--', linewidth=2)
    
        
     
        # Set the y-ticks to correspond to the states and days
        plt.yticks([state['height'] + i * day_offset for i in range(len(chunk_durations_split['start_time'].dt.date.unique())) for state in state_mapping.values()],
                   [f'{i} - {state}' for i in (chunk_durations_split['start_time'].dt.date.unique()) for state in state_mapping.keys()])
        plt.xlabel('Time (Hours)')
        # plt.ylabel('State / Day')
        plt.title(f'Patient ({id_pick}) chunk_{chunk_n}')
        plt.xlim(0, 24)
        plt.xticks(range(25))  # Show every hour on the x-axis
        
        plt.savefig(folder_path + f'/patient_{id_pick}_chunk_{chunk_n}.png')
        plt.clf()

0f352
16f4b
1fbe4
30a32
55cd4
76230
93c14
96adf
a2849
b0455
c55f8
c5785
c8574
d7a46
e2472
ec812
f220c


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

...

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>